# ÉTAPE 1 — Créer les mel-spectrogrammes (TP7 : `Wav_To_melSpect_imgs` + `moveTrain` / `moveTest`)

Chaque son `.wav` (PASCAL + PhysioNet) devient une **image PNG** de son mel-spectrogramme,
puis les images sont rangées en dossiers pour le K-Fold :
```
data/images/
  all/normal/*.png  all/abnormal/*.png     <- toutes les images
  train_kfold1/  test_kfold1/              <- fold 1 : apprentissage / validation
  train_kfold2/  test_kfold2/
  train_kfold3/  test_kfold3/
  test/                                    <- 20 % mis de côté : note finale du vote
```

**Avant de lancer :**
1. *Add Input* → ton dataset `safanasri002/dataset` (dossiers `physionet/` et `pascal/`)
2. *(conseillé)* *Add Input* → les résultats du notebook `kaggle_kfold` (contenu de `resultats_kfold.zip`)
   → on vérifie que les sons sont dans le même ordre que pour le SVM (= mêmes folds)
3. Pas besoin de GPU ni d'Internet pour cette étape
4. **Save Version → Save & Run All** : le dossier `data/images` devient la *sortie* du notebook,
   que tu ajouteras en *Input* du notebook de l'étape 2 (`kaggle_cnn.ipynb`)

## 0. Dossiers

In [ ]:
import os, glob, shutil

# On cherche tout seul le dataset ajouté en Input (pas besoin de modifier les chemins) :
#   1. PASCAL + PhysioNet d'origine (dossiers pascal/ et physionet/) -> mêmes sons, mêmes folds que le SVM
#   2. sinon un dataset trié par classe, comme ourdataset/train/normal et abnormal (TP1)
pascal = glob.glob("/kaggle/input/**/pascal", recursive=True)
by_class = [os.path.dirname(d) for d in glob.glob("/kaggle/input/**/normal", recursive=True)
            if os.path.isdir(os.path.join(os.path.dirname(d), "abnormal"))]
if pascal:
    DATASET_PATH = os.path.dirname(pascal[0])
elif by_class:
    DATASET_PATH = by_class[0]
else:
    raise RuntimeError("Aucun dataset trouvé : Add Input -> ton dataset de sons")
print("Dataset :", DATASET_PATH, os.listdir(DATASET_PATH))

os.chdir("/kaggle/working")
os.makedirs("data", exist_ok=True)
if os.path.islink("data/raw"):              # on refait le lien à chaque fois (au cas où il pointait ailleurs)
    os.unlink("data/raw")
os.symlink(DATASET_PATH, "data/raw")        # prepare_data.py lit data/raw
n_wav = len(glob.glob("data/raw/**/*.wav", recursive=True))
print("data/raw ->", os.path.realpath("data/raw"), ":", n_wav, "fichiers .wav")
assert n_wav > 0, "Aucun .wav dans data/raw : vérifier le dataset ajouté en Input"

# (optionnel) features.npz du notebook kaggle_kfold -> vérifier que l'ordre des sons est le même que pour le SVM
for f in glob.glob("/kaggle/input/**/features.npz", recursive=True)[:1]:
    shutil.copy(f, "data/features.npz")
    print("features.npz trouvé :", f)

## 1. `Wav_To_melSpect_imgs` : .wav → image mel-spectrogramme

Réglages du TP7 adaptés à **2000 Hz** (rééchantillonnage imposé par l'énoncé) :

| | TP7 | ici |
|---|---|---|
| fréquence | 44 100 Hz | 2000 Hz (+ 5 s fixes, volume normalisé = comme le SVM) |
| `n_fft` / `hop_length` | 1024 / 256 | 256 / 64 |
| `n_mels`, `power`, `amplitude_to_db` | 40, 1.0, oui | 40, 1.0, oui |
| `fmin` / `fmax` | 20 / 8000 Hz | 20 / 1000 Hz (max possible à 2000 Hz) |
| image | `specshow` + `savefig` (avec marges et axes) | pixels directs 40 × 157 (le serveur pourra refaire la même image) |

In [ ]:
%%writefile prepare_data.py
"""ÉTAPE 1 — Data mining : à lancer UNE SEULE FOIS.

Lit tous les .wav, les nettoie, extrait les MFCC et sauvegarde le résultat
dans data/features.npz. Ensuite, les modèles chargent directement ce fichier.

    python prepare_data.py
"""
import csv
from pathlib import Path

import librosa
import numpy as np

RAW = Path("data/raw")              # où tu mets PASCAL et PhysioNet
OUTPUT = Path("data/features.npz")  # le résultat (petit fichier)


def preprocess(path):
    """Un fichier .wav -> un vecteur de 40 nombres (MFCC)."""
    y, sr = librosa.load(path, sr=2000)            # 1. rééchantillonnage à 2000 Hz
    y = librosa.util.fix_length(y, size=2000 * 5)  # 2. durée fixe : 5 secondes
    y = librosa.util.normalize(y)                  # 3. volume normalisé
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20, n_fft=256)
    return np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)])  # 4. moyenne + écart-type


# Image mel-spectrogramme pour les CNN — réglages du TP7 (Wav_To_melSpect_imgs) adaptés à 2000 Hz :
#   TP7 : sr=44100, n_fft=1024, hop=256, n_mels=40, power=1.0, fmin=20, fmax=8000, amplitude_to_db
#   ici : sr=2000 (imposé par l'énoncé) -> fmax=1000 (moitié de 2000 Hz), n_fft=256, hop=64
N_MELS = 40       # 40 bandes Mel = 40 lignes de l'image (comme le TP7)
N_FFT = 256       # fenêtre de 128 ms à 2000 Hz
HOP = 64          # pas de 32 ms -> 5 s = 157 colonnes
F_MIN, F_MAX = 20, 1000


def mel_image(path):
    """Un fichier .wav -> image couleur (40 x 157 x 3, uint8) du mel-spectrogramme, comme les PNG du TP7.

    Différence avec le TP7 : on dessine directement les pixels (pas de plt.savefig avec marges ni axes)
    -> le serveur peut refaire EXACTEMENT la même image à partir d'un .wav.
    """
    import matplotlib                              # import ici : seuls les CNN en ont besoin

    y, sr = librosa.load(path, sr=2000)            # 1-3 : MÊME nettoyage que preprocess()
    y = librosa.util.fix_length(y, size=2000 * 5)
    y = librosa.util.normalize(y)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS,
                                         power=1.0, fmin=F_MIN, fmax=F_MAX)
    mel_db = librosa.amplitude_to_db(mel, ref=np.max)          # dB : 0 = le plus fort, -80 = le plus faible
    norm = np.clip((mel_db + 80) / 80, 0, 1)                  # entre 0 et 1
    rgb = matplotlib.colormaps["magma"](norm[::-1])[:, :, :3]  # couleurs de specshow, graves en bas
    return (rgb * 255).astype(np.uint8)


def list_files():
    """Renvoie (chemin, label) avec label 0 = normal, 1 = pathologique."""
    # Cas 1 : dataset déjà trié par classe (ourdataset/train/normal et abnormal, créé au TP1)
    if (RAW / "normal").is_dir() and (RAW / "abnormal").is_dir():
        return [(wav, label) for label, cls in ((0, "normal"), (1, "abnormal"))
                for wav in sorted((RAW / cls).glob("*.wav"))]

    # Cas 2 : datasets d'origine PASCAL + PhysioNet
    files = []
    # PASCAL : le label est le nom du dossier (normal / extra / murmur)
    for folder, label in [("normal", 0), ("extra", 1), ("murmur", 1)]:
        for wav in (RAW / "pascal" / folder).glob("*.wav"):
            files.append((wav, label))

    # PhysioNet : le label est dans REFERENCE.csv  (-1 = normal, 1 = anormal)
    for ref in (RAW / "physionet").rglob("REFERENCE.csv"):
        for name, label in csv.reader(open(ref)):
            files.append((ref.parent / f"{name}.wav", 1 if label == "1" else 0))
    return files


if __name__ == "__main__":
    if OUTPUT.exists():
        print("Déjà fait :", OUTPUT, "(supprime-le pour recommencer)")
        exit()

    X, y = [], []
    for path, label in list_files():
        try:
            X.append(preprocess(path))
            y.append(label)
        except Exception:
            print("Fichier ignoré (corrompu) :", path)

    np.savez(OUTPUT, X=np.array(X), y=np.array(y))
    print(f"{len(y)} sons sauvegardés -> normal: {y.count(0)}, pathologique: {y.count(1)}")

In [ ]:
# Essai sur UN son avant de tout lancer : le spectrogramme "brut" (comme specshow) et l'image qui sera sauvegardée
import importlib
import librosa, librosa.display
import numpy as np
import matplotlib.pyplot as plt
import prepare_data
importlib.reload(prepare_data)    # relit prepare_data.py (sinon Python garde l'ancienne version en mémoire)
from prepare_data import list_files, mel_image, N_FFT, HOP, N_MELS, F_MIN, F_MAX

files = list_files()
print(len(files), "sons trouvés")
assert files, "0 son : relancer la cellule « Dossiers » puis la cellule %%writefile prepare_data.py"
path, label = files[0]
y, sr = librosa.load(path, sr=2000)
y = librosa.util.normalize(librosa.util.fix_length(y, size=2000 * 5))
mel_db = librosa.amplitude_to_db(librosa.feature.melspectrogram(
    y=y, sr=sr, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS, power=1.0, fmin=F_MIN, fmax=F_MAX), ref=np.max)

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
img = librosa.display.specshow(mel_db, sr=sr, hop_length=HOP, x_axis="time", y_axis="mel", fmin=F_MIN, fmax=F_MAX, ax=axes[0])
fig.colorbar(img, ax=axes[0], format="%+2.0f dB")
axes[0].set_title(f"specshow : {os.path.basename(path)} (label {label})")
axes[1].imshow(mel_image(path), aspect="auto", extent=[0, 5, 0, N_MELS])
axes[1].set_title(f"image sauvegardée : {mel_image(path).shape}")
plt.tight_layout()
plt.show()

## 2. Création de toutes les images + `moveTrain` / `moveTest` (dossiers K-Fold, + 20 % de test)

In [ ]:
%%writefile prepare_images.py
"""ÉTAPE 1 bis (CNN) — comme le TP7 : .wav -> images PNG du mel-spectrogramme, puis découpage en dossiers K-Fold.

    python prepare_images.py

Résultat (même organisation que le TP7 : un dossier par classe, lisible directement) :
    data/images/
      all/normal/*.png, all/abnormal/*.png          <- toutes les images
      train_kfold1/ test_kfold1/                    <- fold 1 : apprentissage / validation
      train_kfold2/ test_kfold2/
      train_kfold3/ test_kfold3/
      test/                                         <- 20 % mis de côté : note finale du VOTE

Différences avec le TP7 (pour comparer le CNN au SVM de façon juste) :
  - TP7 : KFold(3) sur 100 % des images, le "test" de chaque fold sert aussi de note finale.
  - ici : 20 % mis de côté AVANT (dossier test/), puis StratifiedKFold(3) sur les 80 % restants
          -> EXACTEMENT les mêmes sons que pour le SVM (le découpage ne dépend que des labels y et de random_state=42).
"""
import shutil
from pathlib import Path

import numpy as np
from PIL import Image
from sklearn.model_selection import StratifiedKFold, train_test_split

from prepare_data import list_files, mel_image

OUT = Path("data/images")
CLASSES = {0: "normal", 1: "abnormal"}
N_FOLDS = 3


def wav_to_mel_images():
    """Comme Wav_To_melSpect_imgs du TP7 : un PNG par .wav, rangé dans le dossier de sa classe.
    Le numéro devant le nom garde l'ordre de list_files() -> même ordre que features.npz."""
    names, labels = [], []
    for i, (path, label) in enumerate(list_files()):
        name = f"{CLASSES[label]}/{i:05d}_{Path(path).stem}.png"
        try:
            Image.fromarray(mel_image(path)).save(OUT / "all" / name)
        except Exception:
            print("Fichier ignoré (corrompu) :", path)
            continue
        names.append(name)
        labels.append(label)
    return np.array(names), np.array(labels)


def copy_images(names, folder):
    """Comme moveTrain / moveTest du TP7 : copie les images dans un dossier train/test."""
    for name in names:
        shutil.copy(OUT / "all" / name, OUT / folder / name)


if __name__ == "__main__":
    if OUT.exists():
        print("Déjà fait :", OUT, "(supprime-le pour recommencer)")
        exit()

    # 1. .wav -> PNG
    folders = ["all", "test"] + [f"{s}_kfold{k}" for k in range(1, N_FOLDS + 1) for s in ("train", "test")]
    for folder in folders:
        for cls in CLASSES.values():
            (OUT / folder / cls).mkdir(parents=True, exist_ok=True)
    names, y = wav_to_mel_images()
    print(f"{len(y)} images -> normal: {(y == 0).sum()}, pathologique: {(y == 1).sum()}")

    # Vérification : mêmes labels, même ordre que features.npz (sinon les folds seraient différents du SVM)
    if Path("data/features.npz").exists():
        same = np.array_equal(np.load("data/features.npz")["y"], y)
        print("Même ordre que features.npz :", "OUI" if same else "NON -> les folds ne seront pas les mêmes que le SVM !")

    # 2. 20 % mis de côté pour la note finale (mêmes paramètres que kfold.load_split)
    train_names, test_names, y_train, _ = train_test_split(names, y, test_size=0.2, stratify=y, random_state=42)
    copy_images(test_names, "test")

    # 3. K-Fold stratifié sur les 80 % restants (mêmes paramètres que kfold.kfold_scores)
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    for k, (tr, val) in enumerate(skf.split(train_names, y_train), start=1):
        copy_images(train_names[tr], f"train_kfold{k}")
        copy_images(train_names[val], f"test_kfold{k}")
        print(f"fold {k} : {len(tr)} images d'apprentissage, {len(val)} de validation")
    print(f"test : {len(test_names)} images (note finale)")

In [ ]:
!python prepare_images.py

## 3. Vérification

In [ ]:
from PIL import Image

for d in sorted(os.listdir("data/images")):
    print(f"{d:14s}", {c: len(os.listdir(f"data/images/{d}/{c}")) for c in ("normal", "abnormal")})

fig, axes = plt.subplots(2, 4, figsize=(18, 4.5))
for row, cls in enumerate(["normal", "abnormal"]):
    for col, png in enumerate(sorted(glob.glob(f"data/images/all/{cls}/*.png"))[::400][:4]):
        axes[row][col].imshow(Image.open(png), aspect="auto", extent=[0, 5, 0, 40])
        axes[row][col].set_title(f"{cls} : {os.path.basename(png)}", fontsize=9)
        axes[row][col].set_xlabel("temps (s)")
plt.tight_layout()
plt.show()

## 4. Fin

On retire le lien vers le dataset (sinon Kaggle copierait tous les .wav dans la sortie)
et on zippe les images pour pouvoir aussi les télécharger.

In [ ]:
os.chdir("/kaggle/working")
if os.path.islink("data/raw"):     # peut déjà avoir été retiré si la cellule est relancée
    os.unlink("data/raw")
assert os.path.isdir("data/images/train_kfold1"), "Pas d'images : relancer la cellule !python prepare_images.py"
!rm -f mel_images.zip && cd data && zip -qr ../mel_images.zip images
!ls -lh mel_images.zip
print("-> Save Version, puis dans kaggle_cnn.ipynb : Add Input -> la sortie de ce notebook")